# CalibrateQwen 00: setup, secrets, and data
We validate Hub access and materialize the two conversation datasets used by the training notebooks.

In [1]:
from pathlib import Path

REPO_ROOT = Path('/content/AutoRegressive-Bhasha')
if not REPO_ROOT.exists():
    !git clone https://github.com/ritwikraha/AutoRegressive-Bhasha.git /content/AutoRegressive-Bhasha
%cd /content/AutoRegressive-Bhasha/calibrate_qwen
!pip install -q -r requirements.txt

Cloning into '/content/AutoRegressive-Bhasha'...
remote: Enumerating objects: 171, done.
remote: Counting objects: 100% (171/171), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 171 (delta 74), reused 120 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (171/171), 1.88 MiB | 4.90 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/AutoRegressive-Bhasha/calibrate_qwen
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 117.1 MB/s eta 0:00:00


In [3]:
import os
from google.colab import userdata

os.environ['TINKER_API_KEY'] = userdata.get('TINKER_API_KEY')
os.environ['HF_WRITE_ACCESS'] = userdata.get('HF_WRITE_ACCESS')
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    pass
print('Required Colab secrets loaded.')

Required Colab secrets loaded.


In [6]:
from datasets import load_dataset

REPO_ID = 'ritwikraha/calibrate-qwen-curated'
base = load_dataset(REPO_ID, 'calibrated_mcq', token=os.environ['HF_WRITE_ACCESS'])
teacher = load_dataset(REPO_ID, 'teacher_phase1', token=os.environ['HF_WRITE_ACCESS'])
base, teacher

teacher/issue_phase1_teacher.parquet: reconstructing file:   0%|          |  0.00B / 6.13MB            

teacher/issue_phase1_teacher.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

(DatasetDict({
     train: Dataset({
         features: ['id', 'source', 'source_split', 'source_id', 'subject', 'question', 'choices', 'answer_index', 'answer_label', 'prompt', 'metadata', 'split'],
         num_rows: 9134
     })
     validation: Dataset({
         features: ['id', 'source', 'source_split', 'source_id', 'subject', 'question', 'choices', 'answer_index', 'answer_label', 'prompt', 'metadata', 'split'],
         num_rows: 1000
     })
     test: Dataset({
         features: ['id', 'source', 'source_split', 'source_id', 'subject', 'question', 'choices', 'answer_index', 'answer_label', 'prompt', 'metadata', 'split'],
         num_rows: 1163
     })
     test_ood: Dataset({
         features: ['id', 'source', 'source_split', 'source_id', 'subject', 'question', 'choices', 'answer_index', 'answer_label', 'prompt', 'metadata', 'split'],
         num_rows: 3000
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['id', 'source', 'source_split', 'source_id', 'su

In [9]:
import os
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_WRITE_ACCESS").strip()
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)

from training.prepare_training_data import prepare_training_file

hard_manifest = prepare_training_file(
    output_path="artifacts/training/hard_label_numeric.jsonl",
    repo_id=REPO_ID,
    variant="hard_label",
    confidence_format="numeric",
)

teacher_manifest = prepare_training_file(
    output_path="artifacts/training/teacher_numeric.jsonl",
    repo_id=REPO_ID,
    variant="teacher",
    confidence_format="numeric",
    abstention_threshold=0.6,
)

hard_manifest, teacher_manifest

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


data/train_phase1_2000.parquet: reconstructing file:   0%|          |  0.00B / 2.21MB            

data/train_phase1_2000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

({'repo_id': 'ritwikraha/calibrate-qwen-curated',
  'variant': 'hard_label',
  'confidence_format': 'numeric',
  'abstention_threshold': 0.6,
  'input_records': 2000,
  'written_records': 2000,
  'sources': {'arc_challenge': 211, 'mmlu': 909, 'openbookqa': 880},
  'skipped': {}},
 {'repo_id': 'ritwikraha/calibrate-qwen-curated',
  'variant': 'teacher',
  'confidence_format': 'numeric',
  'abstention_threshold': 0.6,
  'input_records': 2000,
  'written_records': 2000,
  'sources': {'arc_challenge': 211, 'mmlu': 909, 'openbookqa': 880},
  'skipped': {}})